# dbt Lakehouse POC — Interactive Demo

**A guided walkthrough of the data lakehouse proof of concept.**

This notebook runs the full pipeline interactively:
- MSSQL → Parquet extraction
- dbt/DuckDB transformation & testing
- Apache Iceberg output via Nessie catalog
- Interactive analytics & visualisation

> Run each cell in order. Markdown cells contain talking points for presenting.

---
## Setup

Import libraries and configure paths. The Docker containers (MSSQL, LocalStack, Nessie) should already be running via `docker compose up -d`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from dotenv import load_dotenv

# Load .env
load_dotenv()

# Paths
PROJECT_ROOT = Path(".").resolve()
DBT_DIR = PROJECT_ROOT / "dbt_project"
PARQUET_DIR = PROJECT_ROOT / "data" / "parquet"
OUTPUT_DIR = PROJECT_ROOT / "output"

# Ensure output dirs exist
PARQUET_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"dbt directory: {DBT_DIR}")

---
## Part 1: The Problem Statement

> "Most enterprise data pipelines look like this: operational databases feeding directly into reporting tools, or data being manually exported to Excel. There's no versioning, no testing, no reproducibility."
>
> "What we want is a **lakehouse** — a modern architecture that combines the reliability of a data warehouse with the flexibility of a data lake. Open formats, open tools, no vendor lock-in."
>
> "This POC demonstrates the entire journey:"

```
MSSQL (source) → Parquet (extract) → dbt/DuckDB (transform + test) → Iceberg/Nessie (catalog) → Analytics
```

---
## Part 2: The Source System — SQL Server

> "Here's our operational SQL Server — 8 customers, 8 products, 18 orders. Small dataset, but the pattern scales."
>
> "In production this would be any MSSQL source. The extraction step discovers tables automatically."

Let's seed the database and look at the source data:

In [ ]:
# Check MSSQL container is healthy
result = subprocess.run(
    ["docker", "inspect", "--format", "{{.State.Health.Status}}", "lakehouse-mssql"],
    capture_output=True, text=True
)
print(f"MSSQL container status: {result.stdout.strip()}")

In [ ]:
# Seed the database
sa_password = os.environ.get("SA_PASSWORD", "YourStrong!Passw0rd")

result = subprocess.run([
    "docker", "exec", "lakehouse-mssql",
    "/opt/mssql-tools18/bin/sqlcmd",
    "-S", "localhost", "-U", "SA", "-P", sa_password, "-No",
    "-i", "/scripts/init-db.sql"
], capture_output=True, text=True)

if result.returncode == 0:
    print("✓ Database seeded successfully")
else:
    print(f"Seed output: {result.stdout}")
    if result.stderr:
        print(f"Errors: {result.stderr}")

In [ ]:
# Query the source data — customers
import pyodbc
import pandas as pd

conn_str = (
    f"DRIVER={{{os.environ.get('MSSQL_DRIVER', 'ODBC Driver 18 for SQL Server')}}};"
    f"SERVER={os.environ.get('MSSQL_SERVER', 'localhost')},{os.environ.get('MSSQL_PORT', '1433')};"
    f"DATABASE={os.environ.get('MSSQL_DATABASE', 'lakehouse_source')};"
    f"UID={os.environ.get('MSSQL_USER', 'SA')};"
    f"PWD={os.environ.get('MSSQL_PASSWORD', sa_password)};"
    f"TrustServerCertificate={'yes' if os.environ.get('MSSQL_TRUST_CERT', '1') == '1' else 'no'}"
)

conn = pyodbc.connect(conn_str)
print("✓ Connected to MSSQL")

# Show customers
df_customers = pd.read_sql("SELECT * FROM dbo.customers", conn)
print(f"\nCustomers ({len(df_customers)} rows):")
df_customers

In [ ]:
# Show orders summary
df_orders = pd.read_sql("""
    SELECT COUNT(*) as total_orders,
           COUNT(DISTINCT customer_id) as unique_customers,
           COUNT(DISTINCT product_id) as unique_products,
           MIN(order_date) as earliest_order,
           MAX(order_date) as latest_order
    FROM dbo.orders
""", conn)
df_orders

---
## Part 3: Extract to Parquet

> "We pull the data out of SQL Server into **Apache Parquet** — a columnar, compressed, schema-embedded format."
>
> "Why Parquet? It's the lingua franca of the data ecosystem. DuckDB, Spark, Pandas, Polars — everything reads it. And it decouples us from the source system. dbt never touches SQL Server directly."
>
> "This is your **clean handoff boundary**."

Running the extraction:

In [ ]:
# Run extract.py
result = subprocess.run([sys.executable, "extract.py"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0 and result.stderr:
    print(f"Errors: {result.stderr}")

# Show extracted files
print("\nExtracted Parquet files:")
for f in sorted(PARQUET_DIR.glob("*.parquet")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:30s} {size_kb:.1f} KB")

In [ ]:
# Peek at the Parquet data with DuckDB
import duckdb

con = duckdb.connect()
for f in sorted(PARQUET_DIR.glob("*.parquet")):
    table_name = f.stem
    row_count = con.execute(f"SELECT COUNT(*) FROM '{f}'").fetchone()[0]
    cols = con.execute(f"DESCRIBE SELECT * FROM '{f}'").fetchall()
    col_names = [c[0] for c in cols]
    print(f"{table_name}: {row_count} rows, columns: {', '.join(col_names)}")
con.close()

---
## Part 4: Transform with dbt

> "dbt is the transformation layer. Three layers:"
> - **Staging** — views that type-cast and clean the raw data. No business logic, just shaping.
> - **Marts** — enriched fact table. Orders joined with customers and products, computed metrics.
> - **Reporting** — pre-aggregated tables: revenue by country, category, customer summaries, product performance.
>
> "Everything is SQL. Version-controlled. Reviewable in a PR. No black-box ETL tools."
>
> "The execution engine is **DuckDB** — an in-process analytical database. No server, no cluster."

Running dbt:

In [ ]:
# Run dbt
result = subprocess.run(
    ["dbt", "run", "--profiles-dir", "."],
    cwd=str(DBT_DIR),
    capture_output=True, text=True,
    env={**os.environ, "PATH": os.environ["PATH"]}
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0 and result.stderr:
    print(f"Errors: {result.stderr[-1000:]}")

In [ ]:
# Query the dbt output — enriched orders mart
con = duckdb.connect(str(OUTPUT_DIR / "analytics.duckdb"), read_only=True)

df_enriched = con.execute("SELECT * FROM orders_enriched LIMIT 10").df()
print(f"orders_enriched: {con.execute('SELECT COUNT(*) FROM orders_enriched').fetchone()[0]} rows")
df_enriched

In [ ]:
# Revenue by country
df_revenue = con.execute("SELECT * FROM rpt_revenue_by_country ORDER BY total_revenue DESC").df()
print("Revenue by Country:")
df_revenue

In [ ]:
# Customer summary
df_customers = con.execute("SELECT * FROM rpt_customer_summary ORDER BY total_spent DESC").df()
print("Customer Summary:")
df_customers

---
## Part 5: Test the Data

> "26 data quality tests — uniqueness constraints, not-null checks, referential integrity."
>
> "This is your **data contract layer**. If someone changes the source schema, or bad data sneaks in, the pipeline catches it here — not in a dashboard three weeks later."
>
> "Tests are defined in YAML alongside the models. Same repo, same PR, same review process."

Running dbt tests:

In [ ]:
# Run dbt tests
result = subprocess.run(
    ["dbt", "test", "--profiles-dir", "."],
    cwd=str(DBT_DIR),
    capture_output=True, text=True,
    env={**os.environ, "PATH": os.environ["PATH"]}
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)

---
## Part 6: Load to Apache Iceberg

> "Now we write the output into **Apache Iceberg** — an open table format with warehouse-grade features:"
> - **ACID transactions** — no partial writes
> - **Schema evolution** — add/rename/drop columns without rewriting data
> - **Time travel** — query any previous version of a table
> - **Partition pruning** — only read what you need
>
> "The catalog is **Apache Nessie** — like Git for your data catalog. Branches, experiments, merge when ready."
>
> "This is the **open lakehouse pattern**: any tool that speaks Iceberg REST can read this data."

Loading to Iceberg:

In [ ]:
# Check Nessie is reachable
import requests

nessie_url = os.environ.get("NESSIE_URL", "http://localhost:19120")
try:
    resp = requests.get(f"{nessie_url}/api/v2/config", timeout=5)
    print(f"✓ Nessie catalog is reachable at {nessie_url}")
    print(f"  Version: {resp.json().get('defaultBranch', 'unknown')}")
except Exception as e:
    print(f"✗ Nessie not reachable: {e}")

In [ ]:
# Run iceberg_output.py
result = subprocess.run([sys.executable, "iceberg_output.py"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0 and result.stderr:
    print(f"Errors: {result.stderr}")

In [ ]:
# Verify Iceberg tables in the Nessie catalog
from pyiceberg.catalog.rest import RestCatalog

NESSIE_URL = os.environ.get("NESSIE_URL", "http://localhost:19120/api/v1")
S3_ENDPOINT = os.environ.get("S3_ENDPOINT", "http://localhost:4566")

catalog = RestCatalog(
    name="nessie",
    uri=NESSIE_URL if "/api/" in NESSIE_URL else f"{NESSIE_URL}/api/v1",
    **{
        "s3.endpoint": S3_ENDPOINT,
        "s3.access-key-id": "test",
        "s3.secret-access-key": "test",
    }
)

namespaces = catalog.list_namespaces()
print(f"Namespaces: {namespaces}")

for ns in namespaces:
    tables = catalog.list_tables(ns)
    print(f"\nTables in {ns}:")
    for t in tables:
        tbl = catalog.load_table(t)
        print(f"  {t[1]:35s} — {len(tbl.schema().fields)} columns")

---
## Part 7: Interactive Analytics via Iceberg

> "Here's the payoff. An analyst can connect to the Iceberg catalog and query directly — no ETL team in the loop."
>
> "The data is accessible in an open format through a standard API. Power BI, Tableau, Spark — they can all connect to the same catalog."

### Schema Inspection

In [ ]:
# Inspect Iceberg table schemas
TABLE_NAMES = [
    "orders_enriched",
    "rpt_revenue_by_country",
    "rpt_revenue_by_category",
    "rpt_customer_summary",
    "rpt_product_performance",
]

NAMESPACE = namespaces[0] if namespaces else ("lakehouse",)

for name in TABLE_NAMES:
    try:
        tbl = catalog.load_table(f"{'.'.join(NAMESPACE)}.{name}")
        print(f"\n{'='*60}")
        print(f"TABLE: {name}")
        print(f"{'='*60}")
        for field in tbl.schema().fields:
            print(f"  {field.name:30s} {str(field.field_type):20s} {'(optional)' if field.optional else '(required)'}")
    except Exception as e:
        print(f"  Could not load {name}: {e}")

### Time Travel — Snapshot History

> "Every write to an Iceberg table creates an immutable snapshot. We can browse the full history."

In [ ]:
# Show snapshot history
from datetime import datetime, timezone

for name in TABLE_NAMES:
    try:
        tbl = catalog.load_table(f"{'.'.join(NAMESPACE)}.{name}")
        snapshots = tbl.metadata.snapshots
        print(f"\n{name} — {len(snapshots)} snapshot(s):")
        for snap in snapshots:
            ts = datetime.fromtimestamp(snap.timestamp_ms / 1000, tz=timezone.utc)
            print(f"  ID: {snap.snapshot_id}  |  {ts.strftime('%Y-%m-%d %H:%M:%S UTC')}  |  {snap.summary}")
    except Exception as e:
        print(f"  {name}: {e}")

### Visualisations

Revenue breakdown and customer analytics from the Iceberg tables:

In [ ]:
import matplotlib.pyplot as plt
import plotly.express as px

def load_table(name):
    tbl = catalog.load_table(f"{'.'.join(NAMESPACE)}.{name}")
    return tbl.scan().to_pandas()

# Revenue by country
df_rev_country = load_table("rpt_revenue_by_country")
fig = px.bar(
    df_rev_country.groupby("country")["total_revenue"].sum().reset_index(),
    x="country", y="total_revenue",
    title="Total Revenue by Country",
    labels={"total_revenue": "Revenue", "country": "Country"},
    color="country"
)
fig.update_layout(showlegend=False)
fig.show(renderer="notebook")

In [ ]:
# Revenue by category
df_rev_cat = load_table("rpt_revenue_by_category")
fig = px.pie(
    df_rev_cat,
    values="total_revenue", names="category",
    title="Revenue by Product Category"
)
fig.show(renderer="notebook")

In [ ]:
# Customer summary — top spenders
df_cust = load_table("rpt_customer_summary")
fig = px.bar(
    df_cust.sort_values("total_spent", ascending=True),
    x="total_spent", y="customer_name",
    orientation="h",
    title="Customer Spend (Top to Bottom)",
    labels={"total_spent": "Total Spent", "customer_name": "Customer"}
)
fig.show(renderer="notebook")

In [ ]:
# Product performance
df_prod = load_table("rpt_product_performance")
fig = px.scatter(
    df_prod,
    x="total_units_sold", y="total_revenue",
    size="num_orders", hover_name="product_name",
    title="Product Performance: Units vs Revenue",
    labels={"total_units_sold": "Units Sold", "total_revenue": "Revenue"}
)
fig.show(renderer="notebook")

---
## Part 8: The Full Picture

```
MSSQL (source) → Parquet (extract) → dbt/DuckDB (transform + test) → Iceberg/Nessie (catalog) → Analytics
```

### What we've built:

1. **Extraction** that decouples us from the source system
2. **Transformations** that are version-controlled, tested, and reviewable
3. **An open table format** that any tool can read — no vendor lock-in
4. **A catalog** that tracks schema evolution and supports branching
5. **Self-serve analytics** without needing a dedicated warehouse

> The entire stack is open-source. It runs on a laptop. No cloud dependencies, no licence costs, no proprietary formats.

### For production:
- Swap LocalStack for real S3/ADLS
- Add scheduling (Airflow, cron, CI)
- Point BI tools (Power BI, Tableau) at the Nessie catalog
- The architecture stays the same.

---

### Common Questions

| Question | Answer |
|----------|--------|
| **Why not Spark?** | DuckDB handles analytical workloads without cluster overhead. Iceberg output is Spark-compatible if you scale later. |
| **vs Databricks/Snowflake/Synapse?** | Same concept, fully open-source. No vendor lock-in. Migrate to any platform later — everything is open formats. |
| **Incremental loads?** | dbt supports incremental materialisation. Extract can use CDC. Iceberg handles upserts natively. |
| **Power BI / Tableau?** | Yes — via DuckDB ODBC/JDBC or Trino/Spark connected to the Nessie catalog. |
| **Governance / lineage?** | dbt generates lineage graphs. Nessie versions every schema change. Add DataHub/OpenMetadata for full governance. |

In [ ]:
# Cleanup
try:
    con.close()
except:
    pass
try:
    conn.close()
except:
    pass
print("✓ Demo complete!")